# 00 — Data Preparation (ABCD Release 6)

Load R6 parquets, merge, recode, QC-filter, and save baseline dataset for regression notebooks.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.core.config import initialize_notebook
env = initialize_notebook(run_name="regression", regenerate_run_id=True)

## 1. Run preprocessing pipeline

In [ ]:
from src.core.preprocessing.pipeline import preprocess_abcd_data

train, val, test = preprocess_abcd_data(env)
print(f"\nTrain: {len(train)}, Val: {len(val)}, Test: {len(test)}")
print(f"Total: {len(train) + len(val) + len(test)}")

## 2. Inspect merged baseline data

In [ ]:
import pandas as pd
import numpy as np

full_df = pd.concat([train, val, test], ignore_index=True)
print(f"Columns: {full_df.shape[1]}")
print(f"Subjects: {full_df.shape[0]}")
full_df.head()

## 3. Verify dopamine ROI features

In [ ]:
from src.core.features import get_roi_columns_from_config

dopa_cols = get_roi_columns_from_config(env.configs.data, ["dopamine_core"])
present = [c for c in dopa_cols if c in full_df.columns]
missing = [c for c in dopa_cols if c not in full_df.columns]

print(f"Dopamine features: {len(dopa_cols)} defined, {len(present)} present, {len(missing)} missing")
if missing:
    print(f"  Missing: {missing}")

# Summary stats for present features
full_df[present].describe().round(2)

## 4. Verify ICV column

In [ ]:
icv_col = env.configs.data.get("icv_column")
print(f"ICV column: {icv_col}")

if icv_col and icv_col in full_df.columns:
    print(f"  Present: {full_df[icv_col].notna().sum()} / {len(full_df)} non-null")
    print(f"  Mean: {full_df[icv_col].mean():.0f}, Std: {full_df[icv_col].std():.0f}")

    # Sanity check: ICV should correlate with subcortical volumes
    from scipy.stats import pearsonr
    vol_cols = [c for c in present if "__vol__" in c]
    print(f"\nICV correlations with subcortical volumes:")
    for c in vol_cols:
        mask = full_df[[icv_col, c]].notna().all(axis=1)
        if mask.sum() > 10:
            r, p = pearsonr(full_df.loc[mask, icv_col], full_df.loc[mask, c])
            short_name = c.split("__")[-2] if "__" in c else c
            print(f"  {short_name:<15} r = {r:.3f}")
else:
    print("  WARNING: ICV column not found in data!")

## 4b. dMRI QC statistics

In [ ]:
# dMRI recommended inclusion indicator
dmri_col = "mr_y_qc__incl__dmri_indicator"
if dmri_col in full_df.columns:
    dmri_pass = full_df[dmri_col].astype(str) == "1"
    dmri_fail = ~dmri_pass
    print(f"dMRI QC inclusion:")
    print(f"  Pass: {dmri_pass.sum()} ({dmri_pass.mean()*100:.1f}%)")
    print(f"  Fail/Missing: {dmri_fail.sum()} ({dmri_fail.mean()*100:.1f}%)")
    print(f"  Lost subjects for combined dataset: {dmri_fail.sum()}")
else:
    print(f"WARNING: {dmri_col} not found in data")

## 4c. Dataset verification: structural-only vs combined

In [ ]:
from copy import deepcopy
from src.core.features import get_roi_columns_from_config

# Structural-only dataset (no dMRI QC filter needed)
env_struct = deepcopy(env)
env_struct.configs.data["roi_features"]["dopamine_core"]["connectivity"] = []
struct_cols = get_roi_columns_from_config(env_struct.configs.data, ["dopamine_core"])
print(f"Structural-only features: {len(struct_cols)}")
print(f"Structural-only N: {len(full_df)}")

# Combined dataset (structural + DTI, requires dMRI QC pass)
combined_cols = get_roi_columns_from_config(env.configs.data, ["dopamine_core"])
dmri_col = "mr_y_qc__incl__dmri_indicator"
combined_df = full_df[full_df[dmri_col].astype(str) == "1"].copy()
print(f"\nCombined features: {len(combined_cols)}")
print(f"Combined N (dMRI QC pass): {len(combined_df)}")
print(f"\nSubjects lost for DTI: {len(full_df) - len(combined_df)}")

## 4d. Verify ICV ratio correction

In [ ]:
from src.core.preprocessing.tbv_correction import (
    apply_icv_ratio_correction, identify_volume_features
)

# Demonstrate ratio correction on sample data
vol_indices = identify_volume_features(struct_cols, "__vol__")
X_sample = full_df[struct_cols].values[:5].astype(float)
icv_sample = full_df[icv_col].values[:5].astype(float)

X_ratio = apply_icv_ratio_correction(X_sample, icv_sample, vol_indices)

print("ICV ratio correction verification (first 5 subjects):")
print(f"  Raw caudate_lh range: {X_sample[:, 0].min():.0f} - {X_sample[:, 0].max():.0f}")
print(f"  Ratio caudate_lh range: {X_ratio[:, 0].min():.6f} - {X_ratio[:, 0].max():.6f}")
print(f"  ICV range: {icv_sample.min():.0f} - {icv_sample.max():.0f}")

# Verify AI is unchanged by ratio correction
L_raw = X_sample[:, 0]  # caudate LH
R_raw = X_sample[:, 1]  # caudate RH
L_ratio = X_ratio[:, 0]
R_ratio = X_ratio[:, 1]

ai_raw = (L_raw - R_raw) / (L_raw + R_raw)
ai_ratio = (L_ratio - R_ratio) / (L_ratio + R_ratio)
print(f"\nAI invariance check (caudate):")
print(f"  Max |AI_raw - AI_ratio|: {np.abs(ai_raw - ai_ratio).max():.2e}")
print(f"  AI unchanged by ratio correction: {np.allclose(ai_raw, ai_ratio)}")

# Total features DO change
total_raw = L_raw + R_raw
total_ratio = L_ratio + R_ratio
print(f"\nTotal feature change (caudate):")
print(f"  Raw total: {total_raw.mean():.0f}")
print(f"  Ratio total: {total_ratio.mean():.6f}")

## 5. Target variable (PQ-BC severity) distribution

In [ ]:
import matplotlib.pyplot as plt

target_col = env.configs.regression["targets"][0]["column"]  # pps_severity
y = full_df[target_col].dropna()

print(f"Target: {target_col}")
print(f"  N with data: {len(y)} / {len(full_df)}")
print(f"  Mean: {y.mean():.1f}, Median: {y.median():.1f}, Std: {y.std():.1f}")
print(f"  Range: [{y.min()}, {y.max()}]")

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(y, bins=50, edgecolor="black", alpha=0.7)
ax.set_xlabel("PQ-BC Severity Score")
ax.set_ylabel("Count")
ax.set_title(f"PQ-BC Severity Distribution (n={len(y)})")
ax.axvline(30, color="red", linestyle="--", label="Bin filter lower bound")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Demographics

In [ ]:
print("Sex distribution:")
if "sex_mapped" in full_df.columns:
    print(full_df["sex_mapped"].value_counts())

age_col = env.configs.data["columns"]["mapping"]["age"]
if age_col in full_df.columns:
    age_years = full_df[age_col]  # already in decimal years
    print(f"\nAge: {age_years.mean():.1f} +/- {age_years.std():.1f} years")
    print(f"  Range: [{age_years.min():.1f}, {age_years.max():.1f}]")

# Family structure
family_col = env.configs.data["columns"]["mapping"].get("family_id")
if family_col in full_df.columns:
    n_families = full_df[family_col].nunique()
    fam_sizes = full_df[family_col].value_counts()
    print(f"\nFamilies: {n_families}")
    print(f"  Singletons: {(fam_sizes == 1).sum()}")
    print(f"  Siblings: {(fam_sizes > 1).sum()} families ({(fam_sizes > 1).sum() * 2}+ subjects)")

## 7. Bilateral pairs check

In [ ]:
from src.core.regression.univariate import extract_bilateral_pairs

pairs, unpaired = extract_bilateral_pairs(env.configs.data, ["dopamine_core"])
print(f"Bilateral pairs: {len(pairs)}")
for name, left, right in pairs:
    l_ok = left in full_df.columns
    r_ok = right in full_df.columns
    status = "OK" if (l_ok and r_ok) else "MISSING"
    print(f"  {name:<20} L={l_ok} R={r_ok}  [{status}]")

## 8. Longitudinal data availability

In [ ]:
from src.core.preprocessing.ingest import load_and_merge
from src.core.preprocessing.splits import timepoint_split

# Reload merged data (all timepoints) to check longitudinal availability
merged_all = load_and_merge(env)

tp_col = env.configs.data["columns"]["mapping"]["timepoint"]
id_col = env.configs.data["columns"]["mapping"]["id"]

print("Timepoint counts:")
print(merged_all[tp_col].value_counts().sort_index())

# How many subjects have both baseline and year 2 target?
baseline_tp = env.configs.data["timepoints"]["baseline"]
year2_tp = env.configs.data["timepoints"]["year2"]

bl_ids = set(merged_all[merged_all[tp_col] == baseline_tp][id_col])
y2_ids = set(merged_all[merged_all[tp_col] == year2_tp][id_col])
both = bl_ids & y2_ids

print(f"\nBaseline subjects: {len(bl_ids)}")
print(f"Year 2 subjects: {len(y2_ids)}")
print(f"Both timepoints: {len(both)}")

# Check target availability at year 2
y2_with_target = merged_all[
    (merged_all[tp_col] == year2_tp) & merged_all[target_col].notna()
][id_col].nunique()
print(f"Year 2 with PPS data: {y2_with_target}")

## 9. CBCL Anxiety distribution

In [ ]:
anx_col = 'mh_p_cbcl__dsm__anx_sum'
if anx_col in full_df.columns:
    anx = full_df[anx_col].dropna()
    print(f"CBCL Anxiety (DSM): {anx_col}")
    print(f"  N with data: {len(anx)} / {len(full_df)}")
    print(f"  Mean: {anx.mean():.2f}, Median: {anx.median():.1f}, Std: {anx.std():.2f}")
    print(f"  Range: [{anx.min()}, {anx.max()}]")
    print(f"  Zero-count: {(anx == 0).sum()} ({(anx == 0).mean()*100:.1f}%)")
    print(f"  Non-zero: {(anx > 0).sum()} ({(anx > 0).mean()*100:.1f}%)")

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(anx, bins=50, edgecolor='black', alpha=0.7, color='teal')
    ax.set_xlabel('CBCL DSM Anxiety Sum')
    ax.set_ylabel('Count')
    ax.set_title(f'CBCL Anxiety Distribution (n={len(anx)})')
    plt.tight_layout()
    plt.show()
else:
    print(f"WARNING: {anx_col} not found — check data.yaml columns.metadata")

## 10. Cortical thickness availability

In [ ]:
# Cortical thickness (Desikan atlas) — cortical_dopamine network
cort_cols = get_roi_columns_from_config(env.configs.data, ['cortical_dopamine'])
print(f"Cortical dopamine network: {len(cort_cols)} features defined")

cort_present = [c for c in cort_cols if c in full_df.columns]
cort_missing = [c for c in cort_cols if c not in full_df.columns]
print(f"  Present in data: {len(cort_present)}")
if cort_missing:
    print(f"  Missing: {cort_missing}")

if cort_present:
    nan_counts = full_df[cort_present].isna().sum()
    print(f"\nNaN counts per cortical thickness region:")
    for col in cort_present:
        n_nan = full_df[col].isna().sum()
        pct = n_nan / len(full_df) * 100
        short = col.split('__')[-2] + '_' + col.split('__')[-1]
        print(f"  {short:<25} {n_nan:>5} ({pct:.1f}%)")

    print(f"\nCortical thickness summary (present regions):")
    print(full_df[cort_present].describe().round(3).to_string())

## 11. Year 4 data availability

In [ ]:
# Year 4 availability
year4_tp = env.configs.data['timepoints']['year4']
y4_ids = set(merged_all[merged_all[tp_col] == year4_tp][id_col])
bl_and_y4 = bl_ids & y4_ids

print(f"Year 4 subjects: {len(y4_ids)}")
print(f"Baseline + Year 4: {len(bl_and_y4)}")

# Year 4 target availability
y4_with_pps = merged_all[
    (merged_all[tp_col] == year4_tp) & merged_all[target_col].notna()
][id_col].nunique()
print(f"Year 4 with PPS data: {y4_with_pps}")

# Year 4 anxiety availability
if anx_col in merged_all.columns:
    y4_with_anx = merged_all[
        (merged_all[tp_col] == year4_tp) & merged_all[anx_col].notna()
    ][id_col].nunique()
    print(f"Year 4 with CBCL anxiety data: {y4_with_anx}")

# Year 4 imaging availability (subcortical volumes)
sample_img_col = 'mr_y_smri__vol__aseg__cd__lh_sum'
if sample_img_col in merged_all.columns:
    y4_with_img = merged_all[
        (merged_all[tp_col] == year4_tp) & merged_all[sample_img_col].notna()
    ][id_col].nunique()
    print(f"Year 4 with subcortical imaging: {y4_with_img}")

# Comparison table
print(f"\n{'Timepoint':<12} {'N_subjects':>12} {'PPS_available':>15}")
print('-' * 42)
for label, tp in [('Baseline', baseline_tp), ('Year 2', year2_tp), ('Year 4', year4_tp)]:
    n_sub = merged_all[merged_all[tp_col] == tp][id_col].nunique()
    n_pps = merged_all[
        (merged_all[tp_col] == tp) & merged_all[target_col].notna()
    ][id_col].nunique()
    print(f"{label:<12} {n_sub:>12} {n_pps:>15}")